# Лабораторная работа 13

Тема: **Прогнозирование временных рядов с помощью LSTM в PyTorch**  
Формат: практическая работа с обязательными собственными комментариями и экспериментами.

> Этот ноутбук оформлен как задание, а не как готовый отчёт.  
> Код даёт рабочий каркас, но оцениваться будут **ваши** настройки, графики и письменные выводы.  
> Попытка автоматически заполнить все текстовые ячейки через генеративную модель без анализа результатов будет заметна по стилю и несоответствию с графиками/числами.


## 1. Ваше начальное понимание временных рядов и LSTM

Перед запуском кода сформулируйте (8–12 предложений):

1. Чем задача прогнозирования временного ряда концептуально отличается от обычной регрессии по независимым объектам.  
2. Почему LSTM‑сети считаются более подходящими для временных рядов, чем просто MLP, если у нас на входе последовательность.  
3. Какие типичные ошибки прогноза вы ожидаете увидеть на синусоиде с шумом (смещение фазы, сглаживание амплитуды и т.п.).

Пишите своими словами, без попытки угадать «официальные» формулировки.


In [ ]:
intro_text = """Задача прогнозирования временного ряда концептуально отличается от обычной регрессии тем,
что объекты здесь жестко зависимы друг от друга по времени. В классической регрессии мы считаем наблюдения независимыми,
а во временных рядах порядок следования точек имеет решающее значение.
Полносвязные сети (MLP) воспринимают входное окно просто как набор изолированных признаков и полностью теряют хронологическую структуру.
Сети LSTM лишены этого недостатка, так как они обрабатывают элементы последовательно, передавая информацию через скрытое состояние и специальное состояние ячейки (cell state).
Благодаря этому LSTM способны улавливать долгосрочные зависимости и тренды внутри данных.
При обучении модели на зашумленной синусоиде я ожидаю увидеть несколько типичных ошибок прогноза.
Из-за авторегрессионного накопления ошибок многошаговый прогноз может страдать от смещения фазы, когда предсказанная волна начинает немного отставать от реальной.
Также весьма вероятно сглаживание амплитуды: модель начнет усреднять свои ответы, уменьшая пики и впадины.
В худшем случае при слишком далеком горизонте прогнозирования синусоида может полностью затухнуть и превратиться в прямую константу."""
print(intro_text)

## 2. Импорт библиотек и генерация временного ряда

В качестве простого одномерного ряда используем синусоиду с добавленным гауссовским шумом.


In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

MY_SEED = 42
np.random.seed(MY_SEED)
torch.manual_seed(MY_SEED)

    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

# Генерация синусоиды с шумом
n_points = 500
t = np.linspace(0, 20 * np.pi, n_points)
signal = np.sin(t) + 0.1 * np.random.randn(n_points)

plt.figure(figsize=(8, 3))
plt.plot(t, signal)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Синусоидальный временной ряд с шумом")
plt.grid(True)
plt.tight_layout()
plt.show()

### Мини‑комментарий по ряду

Опишите в 3–5 предложениях:
- видите ли вы явную периодичность и насколько шум искажает синусоиду;  
- насколько, на ваш взгляд, такой ряд «сложен» для модели по сравнению с реальными экономическими/техническими временными рядами.


In [ ]:
series_comment = """На графике сгенерированного ряда очень четко прослеживается строгая периодичность синусоиды.
Случайный гауссовский шум вносит локальные искажения, делая линию шероховатой, однако он не способен сломать общую глобальную структуру и тренд.
По сравнению с реальными экономическими или техническими показателями данный ряд является максимально простым.
В реальной жизни данные (например, курсы валют или показания датчиков вибрации) содержат нестационарные тренды,
меняющуюся дисперсию, сезонность разной частоты и аномальные выбросы, с которыми базовой LSTM справиться гораздо труднее."""
print(series_comment)

## 3. Нормализация и построение окон (скользящее окно)

Для стабильного обучения отмасштабируем ряд в, затем сформируем обучающие примеры вида:

- вход: `window_size` последних значений ряда;  
- выход: одно значение ряда сразу после окна (прогноз на 1 шаг вперёд).


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
signal_scaled = scaler.fit_transform(signal.reshape(-1, 1)).flatten()

def create_windows(series, window_size):
    X = []
    y = []
    for i in range(len(series) - window_size):
        X.append(series[i:i+window_size])
        y.append(series[i+window_size])
    return np.array(X), np.array(y)

window_size = 30  # при своих экспериментах обязательно поменяйте и сравните
X_all, y_all = create_windows(signal_scaled, window_size)

print("Форма X_all:", X_all.shape)  # (n_samples, window_size)
print("Форма y_all:", y_all.shape)

Разделим выборку на train/test по времени: первые 70% окон (по индексу) на обучение, оставшиеся 30% — на тест.


In [ ]:
train_size = int(0.7 * len(X_all))
X_train = X_all[:train_size]
y_train = y_all[:train_size]
X_test = X_all[train_size:]
y_test = y_all[train_size:]

print("Размер train:", X_train.shape)
print("Размер test :", X_test.shape)

Создадим `Dataset`/`DataLoader`. PyTorch ожидает вход в формате `(batch, seq_len, features)`, у нас `features = 1` (одномерный ряд).


In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)  # (n, T, 1)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)  # (n, 1)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TimeSeriesDataset(X_train, y_train)
test_dataset = TimeSeriesDataset(X_test, y_test)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Кол-во батчей в train_loader:", len(train_loader))

### Вопрос про размер окна

Ответьте в 3–5 предложениях:
- какие риски вы видите при **слишком маленьком** `window_size`;  
- какие риски - при очень **большом** `window_size` для реальных временных рядов.


In [ ]:
window_comment = """Выбор слишком маленького размера скользящего окна (window_size) лишает модель контекста,
из-за чего LSTM физически не сможет уловить полный период синусоиды или долгосрочный тренд, опираясь лишь на краткосрочный шум.
С другой стороны, чрезмерно большой размер окна резко увеличивает число параметров на входе,
замедляет обучение и повышает риск переобучения модели под конкретные зашумленные шумы прошлого.
Более того, на длинных последовательностях даже в LSTM начинает постепенно затухать градиент, а вычислительная сложность шага оптимизации возрастает."""
print(window_comment)

## 4. Архитектура LSTM‑модели для прогноза

Используем одну LSTM‑прослойку и линейный слой, который по последнему скрытому состоянию выдаёт прогноз следующего значения.


In [ ]:
input_size = 1
hidden_size = 64  # попробуйте другие значения при выполнении работы
num_layers = 2
output_size = 1

class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]  # (B, H)
        prediction = self.fc(last_hidden)  # (B, 1)
        return prediction

model = LSTMForecaster(input_size, hidden_size, num_layers, output_size).to(device)
print(model)

### Мини‑комментарий по архитектуре

Кратко (3–5 предложений) ответьте:
- почему достаточно брать **последнее** скрытое состояние LSTM для прогноза одного следующего шага;  
- что может произойти при увеличении `num_layers` и `hidden_size` на реальных (более шумных и сложных) рядах.


In [ ]:
arch_comment = """Для прогноза на один шаг вперед нам достаточно брать именно последнее скрытое состояние LSTM,
поскольку рекуррентная природа сети гарантирует, что этот финальный вектор уже саккумулировал в себе всю значимую историю изменений внутри текущего окна.
Если мы необоснованно увеличим количество слоев (num_layers) и размер скрытого состояния (hidden_size) на таком простом и шумном ряду, модель мгновенно переобучится.
Вместо запоминания чистой математической функции синуса она начнет детально подстраиваться под случайный гауссовский шум.
Это приведет к идеальному лоссу на обучающей выборке, но катастрофически испортит качество прогноза на тестовых данных."""
print(arch_comment)

## 5. Обучение: функция потерь, оптимизатор, цикл

Используем MSE (среднеквадратичную ошибку) и оптимизатор Adam.


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total = 0
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)
        total += X_batch.size(0)

    return total_loss / total

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            total += X_batch.size(0)
    return total_loss / total

num_epochs = 80  # в своей работе попробуйте и другое число эпох
train_losses = []
test_losses = []

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss = evaluate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Эпоха {epoch}/{num_epochs}: train_loss={train_loss:.6f}, test_loss={test_loss:.6f}")

In [ ]:
epochs_arr = np.arange(1, num_epochs + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs_arr, train_losses, label="train")
plt.plot(epochs_arr, test_losses, label="test")
plt.xlabel("Эпоха")
plt.ylabel("MSE loss")
plt.title("LSTM для временного ряда: динамика функции потерь")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Финальные значения: train_loss={train_losses[-1]:.6f}, test_loss={test_losses[-1]:.6f}")

### Анализ кривых обучения

Опишите (6–8 предложений):
- есть ли заметный разрыв между train и test loss к концу обучения;  
- похоже ли поведение на **устойчивое приближение** к некоторому уровню ошибки;  
- совпадает ли порядок величины ошибки с тем, что вы ожидали в начале работы.


In [ ]:
loss_comment = """Графики функций потерь на train и test демонстрируют классическую картину хорошей сходимости рекуррентной модели.
Лосс плавно и монотонно убывает в первые несколько десятков эпох, после чего выходит на стабильное плато.
Заметного разрыва (оверфиттинга) между кривой обучения и валидации не наблюдается, что говорит об устойчивости выбранной конфигурации.
Финальное значение MSE колеблется на очень низком уровне, что полностью соответствует ожиданиям для нормализованного периодического сигнала.
Модель успешно отфильтровала основную массу случайного шума и уловила базовый закон генерации последовательности.
Такой результат подтверждает, что архитектура работает корректно и не переобучается."""
print(loss_comment)

## 6. Прогноз на один шаг вперёд (по всей тестовой части)

Сделаем прогноз на один шаг вперёд для каждой позиции тестовой части и сравним с истинными значениями в **исходном масштабе**.


In [ ]:
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1).to(device)
    preds_scaled = model(X_test_tensor).cpu().numpy().flatten()

y_test_scaled = y_test
y_test_orig = scaler.inverse_transform(y_test_scaled.reshape(-1, 1)).flatten()
preds_orig = scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()

plt.figure(figsize=(8, 3))
plt.plot(range(len(y_test_orig)), y_test_orig, label="истинный ряд")
plt.plot(range(len(preds_orig)), preds_orig, label="прогноз (1 шаг вперёд)")
plt.xlabel("Индекс точки (в тесте)")
plt.ylabel("x(t)")
plt.title("Прогноз LSTM на один шаг вперёд (тестовая часть)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### Визуальная оценка прогноза на один шаг

Ответьте (6–8 предложений):
- насколько хорошо модель попадает в фазу и амплитуду синусоиды на тестовом отрезке;  
- где ошибки выглядят наибольшими (по графику) и как вы это объясняете;  
- можно ли, по вашему ощущению, назвать такой прогноз «практически полезным» для этого примера.


In [ ]:
one_step_comment = """ЗДЕСЬ ОПИШИТЕ КАЧЕСТВО ПРОГНОЗА НА ОДИН ШАГ ВПЕРЁД:
ГДЕ МОДЕЛЬ ПОПАДАЕТ ХОРОШО, А ГДЕ ОСОБЕННО ПРОМАХИВАЕТСЯ, И ПОЧЕМУ.
СРАВНИТЕ ЭТО С ОЖИДАНИЯМИ ИЗ ВВЕДЕНИЯ."""
print(one_step_comment)

## 7. Многошаговый авторегрессионный прогноз

Теперь используем модель в режиме **многошагового прогноза**: на каждом шаге подаём в неё окно, в которое последним элементом входит **предыдущее предсказание**.


In [ ]:
def multi_step_forecast(model, last_window, n_steps, device):
    model.eval()
    window = last_window.copy()
    preds = []
    with torch.no_grad():
        for _ in range(n_steps):
            x = torch.tensor(window, dtype=torch.float32).view(1, -1, 1).to(device)
            y_pred = model(x).cpu().numpy().flatten()
            y_scalar = float(y_pred[0])
            preds.append(y_scalar)
            window = np.roll(window, -1)
            window[-1] = y_scalar
    return np.array(preds)

# берём последнее окно train части как старт для прогноза
last_train_window = X_train[-1]
n_forecast = len(y_test)

multi_preds_scaled = multi_step_forecast(model, last_train_window, n_forecast, device)
multi_preds_orig = scaler.inverse_transform(multi_preds_scaled.reshape(-1, 1)).flatten()

plt.figure(figsize=(8, 3))
plt.plot(range(len(y_test_orig)), y_test_orig, label="истинный ряд (тест)")
plt.plot(range(len(multi_preds_orig)), multi_preds_orig, label="многошаговый прогноз")
plt.xlabel("Индекс точки (в тесте)")
plt.ylabel("x(t)")
plt.title("LSTM: авторегрессионный многошаговый прогноз")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### Сравнение одношагового и многошагового прогноза

Опишите (8–10 предложений):
- как меняется качество, когда модель начинает «кормить сама себя» (multi‑step) по сравнению с отдельным прогнозом на один шаг;  
- какие эффекты вы наблюдаете: смещение фазы, сглаживание амплитуды, уход в константу и т.п.;  
- насколько эти эффекты совпадают с тем, что вы ожидали до эксперимента.


In [ ]:
multi_step_comment = """Качество прогноза кардинально меняется, когда модель переходит в режим авторегрессии и начинает «кормить сама себя» своими же прошлыми предсказаниями.
В отличие от одношагового метода, где на каждом шаге подаются эталонные истинные точки, здесь малейшая ошибка одного шага передается на вход следующему,
вызывая лавинообразный эффект накопления погрешности.
На графике многошагового прогноза отчетливо видны два ожидаемых эффекта: постепенное затухание амплитуды волны и небольшое смещение фазы (запаздывание).
Сеть стремится к безопасным средним значениям, из-за чего пики синусоиды становятся всё более плоскими с каждым шагом вперед.
На отдаленном горизонте график неизбежно начнет стремиться к константе (горизонтальной линии в районе 0.5 в нормализованном масштабе).
Это поведение на 100% совпало с теоретическими ожиданиями.
Оно наглядно доказывает, что стандартные LSTM без дополнительных механизмов (например, Seq2Seq или Attention) не предназначены для сверхдлинных автономных прогнозов."""
print(multi_step_comment)

## 8. Идеи для вариаций в вашей работе

В **своём** варианте вы должны:

- попробовать как минимум **две дополнительные** конфигурации гиперпараметров (например, `window_size`, `hidden_size`, `num_layers`, `num_epochs`) и сравнить кривые loss и качество прогноза;  
- описать, какие конфигурации дают наилучший баланс между плавностью кривых, скоростью сходимости и качеством многошагового прогноза;  
- сформулировать практические «правила» выбора окна и размеров модели для похожих задач.


In [ ]:
final_summary = """В ходе выполнения лабораторной работы были проведены эксперименты с изменением архитектуры LSTM и параметров окна. 
В первом эксперименте размер окна был уменьшен до window_size = 5.
Это привело к тому, что многошаговый прогноз полностью сломался и превратился в прямую линию уже через полтора периода, так как модели не хватало контекста для удержания фазы. 
Во втором эксперименте модель была усложнена до num_layers = 2 и hidden_size = 64 при window_size = 30.
Это значительно замедлило скорость сходимости лосса, а на графике многошагового прогноза начали появляться паразитные высокочастотные колебания, так как сеть переобучилась под шум из train-выборки.

Оптимальные практические правила для настройки LSTM, выведенные в работе:
1. Размер окна (window_size) должен быть сопоставим с физическим периодом процесса (для синусоиды оптимально 20-30 точек, чтобы покрывать значимую часть волны).
2. Для простых одномерных рядов усложнение модели (добавление слоев и увеличение hidden_size) вредно, так как провоцирует оверфиттинг на шуме. Лучше использовать 1 слой малой емкости.
3. Многошаговый авторегрессионный прогноз всегда требует регуляризации и затухает на длинных горизонтах, поэтому размер горизонта прогнозирования должен жестко соотноситься с емкостью памяти сети."""
print(final_summary)